In [15]:
# os 환경 변수 GEMINI API KEY 입력
import os
from dotenv import load_dotenv

load_dotenv()

True

In [16]:
from langgraph.graph import StateGraph, START, END

[Workflow 공식문서](https://docs.langchain.com/oss/python/langgraph/workflows-agents#orchestrator-worker)

# Evaluator-Optimizer

## Model 정의

In [ ]:
model = init_chat_model("gpt-5-nano", temperature=0.7)

## State 정의

In [ ]:
class AdState(TypedDict):
    product_name: str       # 상품명
    ad_copy: str            # 생성된 광고 문구



In [ ]:
from pydantic import BaseModel, Field

class EvaluationResult(BaseModel):


In [ ]:
evaluator_llm

## Node 정의

In [ ]:
# [Generator] 신입 카피라이터
def copywriter_node(state: AdState):
    print(f"\n--- [Copywriter] 광고 문구 작성 중 (시도: {count + 1}) ---")

    if not feedback:
        # 첫 시도는 좀 건조하게 작성하도록 유도 (Fail을 유발하기 위해)
        prompt = f"'{product}'의 기능 위주로 인스타그램 홍보 문구를 건조하게 작성해줘. 홍보 문구만 답변하고 반드시 20자 이하로 작성하시오."
    else:
        # 피드백 반영
        prompt = f"""
        '{product}' 인스타그램 홍보 문구를 다시 작성해.

        <반드시 지켜야 할 수정 사항>
        {feedback}
        </반드시 지켜야 할 수정 사항>

        <작성 시 반드시 지켜야할 사항>
        홍보 문구만 답변하고 절대 50자 이하로 작성하시오.
        </작성 시 반드시 지켜야할 사항>
        """


In [ ]:
# [Evaluator] 감성적인 마케팅 팀장
def manager_node(state: AdState):
    print(f"\n--- [Manager] 문구 검수 중 ---")

    print(f"   ㄴ 신입이 쓴 글: {ad_copy}")

    prompt = f"""
    당신은 깐깐한 마케팅 팀장입니다. 신입 사원이 쓴 다음 광고 문구를 평가하세요:

    "{ad_copy}"

    <평가 기준>
    1. (정량) 해시태그(#)가 3개 이상 있어야 합니다.
    2. (정량) '할인' 또는 '특가'라는 단어가 포함되어야 합니다.
    3. (정성 - 중요!) **문구가 너무 설명문 같거나 딱딱하면 안 됩니다. 소비자의 감성을 자극하는 '활기차고 매력적인 톤'이어야 합니다.**
    </평가 기준>

    위 3가지 기준 중 하나라도 부족하면 fail을 주세요.
    특히 3번(톤앤매너)이 부족하다면 "좀 더 감성적으로 쓰세요" 같이 100자 이내로 조언하세요.
    """


## 그래프 생성

In [ ]:
# 4. 루프 로직
def route_submission(state: AdState):


In [ ]:
# 5. 그래프 생성
workflow = StateGraph(AdState)
workflow.add_node("copywriter_node", copywriter_node)
workflow.add_node("manager_node", manager_node)


In [ ]:
app = workflow.compile()

In [ ]:
app

## 실행

In [ ]:
inputs = {"product_name": "자율주행 자동차"}
result = app.invoke(inputs)

# Orchestrator-Worker

## Model 정의

In [ ]:
model = init_chat_model("gpt-5-nano")

## State 정의

In [ ]:
from typing import List

# 1. 데이터 모델 정의 (Plan & Section)
# 팀장(Orchestrator)이 짤 계획표의 양식
class Section(BaseModel):


In [ ]:
import operator
from typing import Annotated

# 2. State 정의 (Reducer 필수!)
class ReportState(TypedDict):


In [ ]:
# Worker에게 전달할 별도의 State (작업 지시서)
class WorkerState(TypedDict):
    section: Section

In [ ]:
class Plan(BaseModel):
    sections: List[Section] = Field(description="보고서 작성을 위한 목차 리스트")

In [ ]:
planner_llm = model.with_structured_output(Plan)

## Node 정의

In [ ]:
# [Orchestrator] 팀장 노드: 계획 수립
def orchestrator_node(state: ReportState):
    topic = state["topic"]
    print(f"\n--- [Orchestrator] '{topic}' 보고서 계획 수립 중 ---")

    # 보고서 목차를 생성 (최대 3개로 제한하여 속도 조절)
    plan = planner_llm.invoke(f"'{topic}'에 대한 보고서 목차를 짜줘. 3개 섹션 이내로 구성해.")

    print(f"생성된 계획: {[s.name for s in plan.sections]}")
    return {"sections": plan.sections}

In [ ]:
# [Worker] 팀원 노드: 섹션 집필
# 이 노드는 여러 개가 복제되어 동시에 실행
def worker_node(state: WorkerState):
    section = state["section"]
    print(f"   --- [Worker] 집필 중: {section.name} ---")

    prompt = f"""
    다음 섹션에 대한 내용을 짧게 작성해줘.
    제목: {section.name}
    내용 가이드: {section.description}
    """


In [ ]:
# [Synthesizer] 편집자 노드: 취합
def synthesizer_node(state: ReportState):
    print("\n--- [Synthesizer] 모든 원고 취합 및 최종 편집 ---")

    # 리스트에 모인 조각글들을 하나로 합침
    completed = state["completed_sections"]
    final_report = "\n".join(completed)

    return {"final_report": final_report}

## 그래프 생성

### [Send API](https://reference.langchain.com/python/langgraph/types/?h=send#langgraph.types.Send)

In [ ]:
from langgraph.types import Send

# 5. 동적 라우팅 로직 (Map: 1 -> N)
# 일반적인 edge가 아니라, 리스트 개수만큼 노드를 생성해서 뿌려주는 역할
def assign_workers(state: ReportState):


In [ ]:
# 6. 그래프 조립


# 6-1. 시작 -> 팀장


# 6-2. 팀장 -> (Map) -> 팀원들 (Conditional Edge)
workflow.add_conditional_edges(

)

# 6-3. 팀원들 -> (Reduce) -> 편집자


# 6-4. 편집자 -> 종료


In [ ]:
app = workflow.compile()

In [ ]:
app

## 실행

In [ ]:
inputs = {"topic": "생성형 AI의 미래"}
result = app.invoke(inputs)

In [ ]:
result["final_report"]

### develop (화학적 결합)

In [ ]:
# [Synthesizer] 편집자 노드: 단순 취합이 아니라 '화학적 결합'을 수행
def synthesizer_node(state: ReportState):
    topic = state["topic"]
    completed_docs = state["completed_sections"]

    print(f"\n--- [Synthesizer] 원고 {len(completed_docs)}건 도착. 최종 편집 시작 ---")

    # 1. 일단 텍스트 덩어리로 병합


    # 2. LLM에게 '전문 편집자' 역할 부여
    prompt = f"""
    당신은 전문 리포트 편집자입니다.
    다음은 '{topic}'에 대해 여러 작가가 나누어 쓴 원고들입니다.

    이 초안들을 바탕으로 **하나의 자연스럽고 전문적인 보고서**로 다시 작성해주세요.

    [지시사항]
    1. 각 섹션의 연결이 매끄러워야 합니다.
    2. 전체를 아우르는 '서론'과 '결론'을 추가해주세요.
    3. 마크다운(Markdown) 형식을 사용하여 가독성을 높여주세요.

    [원고 내용]
    {raw_content}
    """

    # 최종 생성을 위한 LLM 호출



In [ ]:
# 6. 그래프 조립
workflow = StateGraph(ReportState)

workflow.add_node("orchestrator_node", orchestrator_node)
workflow.add_node("worker_node", worker_node)
workflow.add_node("synthesizer_node", synthesizer_node)

# 1. 시작 -> 팀장
workflow.add_edge(START, "orchestrator_node")

# 2. 팀장 -> (Map) -> 팀원들 (Conditional Edge)
workflow.add_conditional_edges(
    "orchestrator_node",
    assign_workers,
    ["worker_node"] # 이 부분은 생략 가능하지만 명시적으로 적어줌
)

# 3. 팀원들 -> (Reduce) -> 편집자
workflow.add_edge("worker_node", "synthesizer_node")

# 4. 편집자 -> 종료
workflow.add_edge("synthesizer_node", END)

In [ ]:
app = workflow.compile()

In [ ]:
app

In [ ]:
inputs = {"topic": "생성형 AI의 미래"}
result = app.invoke(inputs)

In [ ]:
result["final_report"]